# PyTorch 第二章：核心模块

> 目标：先看懂一套完整训练流程，再理解 Tensor、计算图与 Autograd 为什么能让训练发生。

本 Notebook 结合《PyTorch 实用教程（第二版）》第二章整理，并按当前 PyTorch 2.x 的常用实践做少量简化与校正。

学习顺序：

1. **2.1 模块结构**：知道 `nn / autograd / optim / utils.data` 分别干什么  
2. **2.2 完整训练流程**：数据 → 模型 → Loss → backward → step  
3. **2.3 Tensor**：PyTorch 中的数据载体  
4. **2.4 Tensor 常用操作**：创建、变形、拼接、数学运算  
5. **2.5 计算图**：理解梯度如何沿图传播  
6. **2.6 Autograd**：真正掌握 `backward()`、梯度累积、`grad()`、`detach()` 等

参考：
- https://tingsongyu.github.io/PyTorch-Tutorial-2nd/chapter-2/
- https://docs.pytorch.org/


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch: 2.11.0+cu128
CUDA available: True


## 2.1 PyTorch 模块结构

先只记住四个最常用模块：

| 模块 | 作用 |
|---|---|
| `torch.nn` | 搭建神经网络 |
| `torch.autograd` | 自动求导 |
| `torch.optim` | 根据梯度更新参数 |
| `torch.utils.data` | Dataset / DataLoader |

训练时它们会连成一条链：

`DataLoader → nn.Module → Loss → autograd → optim`


In [3]:
# 看看当前环境中的 torch 安装位置
print(torch.__file__)

# 常见模块
print("nn:", nn)
print("optim:", optim)
print("autograd:", torch.autograd)
print("utils.data:", torch.utils.data)


/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/__init__.py
nn: <module 'torch.nn' from '/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/nn/__init__.py'>
optim: <module 'torch.optim' from '/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/optim/__init__.py'>
autograd: <module 'torch.autograd' from '/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/autograd/__init__.py'>
utils.data: <module 'torch.utils.data' from '/home/yyy/Apps/miniconda3/envs/test/lib/python3.11/site-packages/torch/utils/data/__init__.py'>


## 2.2 先跑通一套完整训练流程

原教程使用 4 张 COVID-19 / 正常胸片展示训练流程。

为了让这个 Notebook 在 Colab **无需额外数据即可运行**，这里用模拟的 `8×8 RGB` 图像代替真实胸片，但保留原教程的核心结构：

- `DataLoader` 提供数据
- `Conv2d → Linear` 构成 TinyCNN
- `CrossEntropyLoss` 计算分类损失
- `SGD` 更新参数
- 训练循环执行：`zero_grad → forward → loss → backward → step`

### 为什么 Linear 输入是 36？

输入为 `8×8`，使用 `3×3` 卷积、`padding=0`：

`8×8 → 6×6`

卷积只输出 1 个通道，因此展平后：

`1 × 6 × 6 = 36`


In [3]:
# ---------- 1. 构造一个可学习的模拟二分类数据集 ----------
# 类别 0：左半边更亮；类别 1：右半边更亮
def make_dataset(n):
    x = torch.randn(n, 3, 8, 8) * 0.3
    y = torch.randint(0, 2, (n,))

    for i in range(n):
        if y[i] == 0:
            x[i, :, :, :4] += 1.0
        else:
            x[i, :, :, 4:] += 1.0
    return x, y

x_train, y_train = make_dataset(128)
x_valid, y_valid = make_dataset(32)

train_loader = DataLoader(
    TensorDataset(x_train, y_train),
    batch_size=16,
    shuffle=True
)

valid_loader = DataLoader(
    TensorDataset(x_valid, y_valid),
    batch_size=32,
    shuffle=False
)

print(x_train.shape, y_train.shape)


torch.Size([128, 3, 8, 8]) torch.Size([128])


In [4]:
# ---------- 2. 模型 ----------
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 1, kernel_size=3)  # 8x8 -> 6x6
        self.fc = nn.Linear(36, 2)                  # 1*6*6 -> 2 类

    def forward(self, x):
        x = torch.relu(self.conv(x))
        x = torch.flatten(x, start_dim=1)
        return self.fc(x)                           # 输出 logits，不需要手动 softmax

model = TinyCNN()
print(model)

# 检查一次 forward 的形状
sample_x, _ = next(iter(train_loader))
print("input :", sample_x.shape)
print("output:", model(sample_x).shape)


TinyCNN(
  (conv): Conv2d(3, 1, kernel_size=(3, 3), stride=(1, 1))
  (fc): Linear(in_features=36, out_features=2, bias=True)
)
input : torch.Size([16, 3, 8, 8])
output: torch.Size([16, 2])


In [5]:
# ---------- 3. Loss + Optimizer ----------
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

# ---------- 4. 训练 ----------
for epoch in range(6):
    model.train()

    total_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()          # 清除上一轮梯度
        logits = model(images)         # forward
        loss = criterion(logits, labels)
        loss.backward()                # 计算梯度
        optimizer.step()               # 真正更新参数

        total_loss += loss.item()

    # 验证阶段：不需要梯度
    model.eval()
    correct = 0
    with torch.inference_mode():
        for images, labels in valid_loader:
            pred = model(images).argmax(dim=1)
            correct += (pred == labels).sum().item()

    acc = correct / len(x_valid)
    print(f"epoch={epoch+1}  loss={total_loss/len(train_loader):.4f}  val_acc={acc:.3f}")


epoch=1  loss=0.4678  val_acc=1.000
epoch=2  loss=0.0229  val_acc=1.000
epoch=3  loss=0.0064  val_acc=1.000
epoch=4  loss=0.0036  val_acc=1.000
epoch=5  loss=0.0024  val_acc=1.000
epoch=6  loss=0.0018  val_acc=1.000


### 训练代码最重要的 5 行

```python
optimizer.zero_grad()
output = model(x)
loss = criterion(output, y)
loss.backward()
optimizer.step()
```

含义：

- `forward`：算预测值
- `loss`：衡量预测和真实标签的差距
- `backward()`：计算梯度
- `step()`：按照梯度修改参数

> `loss.backward()` **不修改权重**；真正修改权重的是 `optimizer.step()`。


## 2.3 Tensor：PyTorch 的核心数据结构

Tensor 可以理解为：**支持 GPU 和自动求导的多维数组**。

常用属性：

| 属性 | 含义 |
|---|---|
| `shape` | 形状 |
| `dtype` | 数据类型 |
| `device` | CPU / GPU |
| `requires_grad` | 是否需要追踪梯度 |
| `grad` | 反向传播得到的梯度 |
| `grad_fn` | 这个 Tensor 是由什么运算产生的 |
| `is_leaf` | 是否为叶子 Tensor |

`torch.Tensor` 是类；日常创建数据更常用 `torch.tensor(...)`。

旧版 PyTorch 的 `Variable` 已不再需要。


In [6]:
# Tensor 的基础属性
x = torch.tensor(
    [[1.0, 2.0],
     [3.0, 4.0]]
)

print("shape :", x.shape)
print("dtype :", x.dtype)
print("device:", x.device)


shape : torch.Size([2, 2])
dtype : torch.float32
device: cpu


In [7]:
# 自动求导相关属性
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)
print("x.grad_fn:", x.grad_fn)
print("y.grad_fn:", y.grad_fn)
print("backward 前 x.grad:", x.grad)

y.backward()

print("backward 后 x.grad:", x.grad)  # dy/dx = 2x = 6


x.is_leaf: True
y.is_leaf: False
x.grad_fn: None
y.grad_fn: <PowBackward0 object at 0x7d9f0a88eef0>
backward 前 x.grad: None
backward 后 x.grad: tensor(6.)


记忆：

- `requires_grad`：**要不要算梯度**
- `grad`：**梯度是多少**
- `grad_fn`：**这个 Tensor 怎么算出来的**
- `is_leaf`：**是不是计算图的叶子节点**

> 不建议把 `.data` 当作日常修改 Tensor 的入口。需要切断计算图时优先使用 `detach()`。


## 2.4 Tensor 高频操作

教程列出的 Tensor API 很多。入门阶段不需要全部背，只掌握下面几类：

1. 创建：`tensor / zeros / ones / arange / linspace / rand / randn`
2. NumPy 转换：`from_numpy`
3. 变形：`reshape / flatten / unsqueeze / squeeze`
4. 拼接：`cat / stack`
5. 数学运算：逐元素运算、矩阵乘法、统计


In [9]:
import numpy as np

# 创建
a = torch.zeros(2, 3)
b = torch.ones(2, 3)
c = torch.arange(0, 6).reshape(2, 3)
d = torch.linspace(0, 1, steps=5)
e = torch.randn(2, 3)

print("zeros:\n", a)
print("arange + reshape:\n", c)
print("linspace:", d)
print("randn:\n", e)


zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])
arange + reshape:
 tensor([[0, 1, 2],
        [3, 4, 5]])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])
randn:
 tensor([[-1.6172, -1.1962, -0.5088],
        [-1.4119,  0.6712,  1.2389]])


In [10]:
# torch.from_numpy 与 NumPy 数组可以共享底层内存
arr = np.array([1, 2, 3], dtype=np.float32)
t = torch.from_numpy(arr)

arr[0] = 99

print("numpy :", arr)
print("tensor:", t)   # tensor 也发生变化


numpy : [99.  2.  3.]
tensor: tensor([99.,  2.,  3.])


In [11]:
x = torch.arange(12).reshape(3, 4)

# 变形
print("原形状:", x.shape)
print("flatten:", torch.flatten(x))
print("增加维度:", x.unsqueeze(0).shape)

# 拼接：cat 沿已有维度拼；stack 创建一个新维度
a = torch.tensor([1, 2])
b = torch.tensor([3, 4])

print("cat  :", torch.cat([a, b], dim=0))
print("stack:", torch.stack([a, b], dim=0))

# 数学运算
m1 = torch.tensor([[1., 2.], [3., 4.]])
m2 = torch.tensor([[2., 0.], [1., 2.]])

print("逐元素乘:\n", m1 * m2)
print("矩阵乘:\n", m1 @ m2)
print("mean:", m1.mean())


原形状: torch.Size([3, 4])
flatten: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
增加维度: torch.Size([1, 3, 4])
cat  : tensor([1, 2, 3, 4])
stack: tensor([[1, 2],
        [3, 4]])
逐元素乘:
 tensor([[2., 0.],
        [3., 8.]])
矩阵乘:
 tensor([[ 4.,  4.],
        [10.,  8.]])
mean: tensor(2.5000)


## 2.5 计算图 Computational Graph

Autograd 的核心不是“背导数公式”，而是：

> PyTorch 在 forward 时记录 Tensor 之间的运算关系，backward 时沿这些关系使用链式法则传播梯度。

用教程中的例子：

$y=(x+w)(w+1)$

拆成：

$a=x+w$

$b=w+1$

$y=a\times b$

当 `x=2, w=1` 时：

$\frac{\partial y}{\partial w}=5$

$\frac{\partial y}{\partial x}=2$


In [12]:
w = torch.tensor([1.0], requires_grad=True)
x = torch.tensor([2.0], requires_grad=True)

a = x + w
b = w + 1
y = a * b

print("is_leaf:")
print("w:", w.is_leaf, "x:", x.is_leaf, "a:", a.is_leaf, "b:", b.is_leaf, "y:", y.is_leaf)

print("\ngrad_fn:")
print("w:", w.grad_fn)
print("a:", a.grad_fn)
print("y:", y.grad_fn)

y.backward()

print("\ngrad:")
print("w.grad =", w.grad)  # 5
print("x.grad =", x.grad)  # 2
print("a.grad =", a.grad)  # 非叶子 Tensor 默认不保留 .grad


is_leaf:
w: True x: True a: False b: False y: False

grad_fn:
w: None
a: <AddBackward0 object at 0x7d9f02acb700>
y: <MulBackward0 object at 0x7da028cd7430>

grad:
w.grad = tensor([5.])
x.grad = tensor([2.])
a.grad = None


/tmp/ipykernel_17136/4269005323.py:21: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:489.)
  print("a.grad =", a.grad)  # 非叶子 Tensor 默认不保留 .grad


### 叶子节点与动态图

- 直接创建并参与求导的参数通常是 **leaf Tensor**
- 运算生成的中间 Tensor 通常是 **non-leaf Tensor**
- 默认情况下，反向传播后主要把梯度累积到需要梯度的叶子 Tensor 的 `.grad`

PyTorch 使用**动态图（dynamic graph）**：

> 运算发生的同时构建计算图，因此普通 Python 控制流可以自然参与模型计算。


## 2.6 Autograd：自动微分

这一节必须真正掌握 4 件事：

1. `backward()`：从输出向叶子节点反向传播
2. **梯度会累积**，不会自动清零
3. `torch.autograd.grad()`：直接求指定输入的导数，可做高阶导数
4. `detach()` / `no_grad()` / `inference_mode()`：控制是否记录计算图


In [13]:
# 1. 梯度默认会累积
w = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = w ** 2
    y.backward()
    print(f"第 {i+1} 次 backward 后 w.grad =", w.grad)
    # 如果不清零：2 -> 4 -> 6

# 手动清零
w.grad.zero_()
print("清零后:", w.grad)


第 1 次 backward 后 w.grad = tensor(2.)
第 2 次 backward 后 w.grad = tensor(4.)
第 3 次 backward 后 w.grad = tensor(6.)
清零后: tensor(0.)


In [14]:
# 2. torch.autograd.grad：一阶、二阶导数
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

# create_graph=True：保留一阶导数对应的计算图，才能继续求二阶导
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
grad2 = torch.autograd.grad(grad1, x)[0]

print("一阶导:", grad1)  # 2x = 6
print("二阶导:", grad2)  # 2


一阶导: tensor(6., grad_fn=<MulBackward0>)
二阶导: tensor(2.)


detach的作用是：从计算图中剥离出“数据”，并以一个新张量的形式返回，并且新张量与旧张量共享数据，简单的可理解为做了一个别名。 请看下例的w，detach后对w_detach修改数据，w同步地被改为了999

In [5]:
w = torch.tensor([1.], requires_grad=True)
x = torch.tensor([2.], requires_grad=True)
a = torch.add(w, x)
b = torch.add(w, 1)
y = torch.mul(a, b)
y.backward()
print(w)
w_detach = w.detach()
w_detach.data[0] = 999
print(w)

tensor([1.], requires_grad=True)
tensor([999.], requires_grad=True)


In [15]:
# 3. detach / no_grad / inference_mode
x = torch.tensor(2.0, requires_grad=True)
y = x * 3

y_detached = y.detach()
print("y.requires_grad         =", y.requires_grad)
print("y_detached.requires_grad=", y_detached.requires_grad)

with torch.no_grad():
    z1 = x * 4

with torch.inference_mode():
    z2 = x * 5

print("no_grad result requires_grad       =", z1.requires_grad)
print("inference_mode result requires_grad=", z2.requires_grad)


y.requires_grad         = True
y_detached.requires_grad= False
no_grad result requires_grad       = False
inference_mode result requires_grad= False


model.eval() 只是在切换模型到评估模式（evaluation mode），主要影响：

Dropout
BatchNorm

它不会关闭 Autograd

### `no_grad()` 与 `inference_mode()`

两者都不会记录 backward 图。

- `torch.no_grad()`：通用，退出后其结果仍更容易与后续 Autograd 代码组合
- `torch.inference_mode()`：更偏向**纯推理**，还能进一步减少 Autograd 相关开销，但限制更严格

模型验证 / 推理通常还要同时：

```python
model.eval()
```

注意：`model.eval()` **不会关闭梯度计算**；它主要改变 Dropout、BatchNorm 等层的行为。


### 选学：自定义 `autograd.Function`

教程 2.6 还介绍了自定义 Function。入门阶段知道结构即可：

```text
forward()  → 定义前向怎么算
backward() → 定义反向梯度怎么算
```

只有在 PyTorch 没有现成算子、或者你需要特殊梯度行为时才常用。


In [ ]:
# 最小自定义 Function 示例：y = exp(x)
class MyExp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        y = x.exp()
        ctx.save_for_backward(y)
        return y

    @staticmethod
    def backward(ctx, grad_output):
        (y,) = ctx.saved_tensors
        return grad_output * y  # d(exp(x))/dx = exp(x)

x = torch.tensor(1.0, requires_grad=True)
y = MyExp.apply(x)
y.backward()

print("y =", y.item())
print("x.grad =", x.grad.item())


# 第二章总结

如果下面这条链已经能自己解释，第二章的核心就掌握了：

`Tensor → forward → 计算图 → loss → backward → grad → optimizer.step()`

必须会回答：

1. `nn / autograd / optim / DataLoader` 分别做什么？
2. 为什么训练前要 `optimizer.zero_grad()`？
3. `loss.backward()` 和 `optimizer.step()` 的区别？
4. `shape / dtype / device` 各表示什么？
5. `requires_grad / grad / grad_fn / is_leaf` 的区别？
6. `cat` 和 `stack` 的区别？
7. 什么是叶子 Tensor？为什么中间 Tensor 的 `.grad` 默认通常是 `None`？
8. 为什么梯度会累积？
9. `detach()` 是什么？
10. `model.eval()` 为什么不能替代 `no_grad()` / `inference_mode()`？

### 建议学习方式

不要一次看完。按下面顺序逐段运行：

**2.1 → 2.2 跑通训练 → 2.3/2.4 熟悉 Tensor → 2.5 手算梯度 → 2.6 对照 Autograd**

当你能够自己写出下面 5 行时，再进入第三章：

```python
optimizer.zero_grad()
output = model(x)
loss = criterion(output, y)
loss.backward()
optimizer.step()
```
